# Что здесь

В данном ноутбуке содержатся эксперименты с прунингом. Эксперименты были проведены для всех моделей whisper с huggingface.

с помощью unstructured magnitude pruning по L1-норме удалось уменьшить RTF всех моделей без потери существенной потери точности по метрике WER и CER.

Так например RTF whisper-large-v3 был уменьшен с $1.313$ до $1.002$

### imports

In [37]:
import json
import time
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

import typing as typ

import pandas as pd

import numpy as np
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

import jiwer
from jiwer import Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip

import typing as typ
import os

from IPython.display import Audio
from IPython.display import clear_output

### constants

In [18]:
# models was sorted by size
WHISPER_MODELS = (
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    "openai/whisper-medium",
    "openai/whisper-large-v3"
)

DEVICE_CPU = "cpu"


WHISPER_SR = 16000
WHISPER_MAX_LABEL_LEN = 256

RANDOM_STATE = 62
MINI_DS_LEN = 50

### geting models

In [19]:
def load_model(model_id: str, device = DEVICE_CPU):
    model = WhisperForConditionalGeneration.from_pretrained(model_id)
    model.generation_config.language = "ru"
    model.generation_config.task = "transcribe"
    processor = WhisperProcessor.from_pretrained(model_id)

    return model, processor

### model measurements

In [20]:
TRANSFORM = jiwer.Compose([
    jiwer.RemovePunctuation(),
    jiwer.ToLowerCase(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])

def calc_wer(references, predictions):
    return jiwer.wer(
        TRANSFORM(references),
        TRANSFORM(predictions),
    )

def calc_cer(references, predictions):
    return jiwer.cer(
        TRANSFORM(references),
        TRANSFORM(predictions),
    )

def process_sample(sample, processor, device="cpu", dtype=None):
    audio_array = sample["audio"]["array"]
    audio_sr = sample["audio"]["sampling_rate"]
    ref = sample["transcription"]

    duration = len(audio_array) / audio_sr

    inputs = processor(
        audio_array,
        sampling_rate=audio_sr,
        return_tensors="pt",
    ).to(device)
    if dtype:
        inputs = inputs.to(dtype=dtype)

    return inputs, duration, ref

In [21]:
def warmup(model, processor, dataset, warmup_iters: int, device):
    with torch.inference_mode():
        for i, sample in enumerate(dataset):
            if i >= warmup_iters:
                break
            
            inputs, _, _ = process_sample(sample, processor, device=device, dtype=model.dtype)
            _ = model.generate(**inputs)

def evaluate(
    model,
    processor,
    dataset,
    warmup_iters: int = 1,
    device = DEVICE_CPU
):
    predictions: typ.List[str] = list()
    references:  typ.List[str] = list()
    
    total_time = 0.0
    total_duration = 0.0
    num_samples = 0

    warmup(model, processor, dataset, warmup_iters, device)
    with torch.inference_mode():
        for i, sample in enumerate(tqdm(dataset)):
            inputs, duration, ref = process_sample(sample, processor, device=device, dtype=model.dtype)

            start = time.perf_counter()
            output_ids = model.generate(**inputs)
            end = time.perf_counter()

            pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]


            total_time += (end - start)
            total_duration += duration
            num_samples += 1

            predictions.append(pred)
            references.append(ref)

    rtf = total_time / total_duration if total_duration > 0 else 0.0
    wer = calc_wer(references, predictions)
    cer = calc_cer(references, predictions)

    return {
        "rtf": rtf,
        "wer": wer,
        "cer": cer,
        "total_duration": total_duration,
        "total_time": total_time,
        "num_samples": num_samples,
    }

In [22]:
val_dataset = load_dataset("bond005/sberdevices_golos_10h_crowd", split="validation")
mini_dataset = val_dataset.shuffle(seed=RANDOM_STATE).select(range(MINI_DS_LEN))

In [ ]:
def print_model_result(model_name: str, result: typ.Dict):
    print(f'===={model_name}====')
    for metric, val in result.items():
        print(f"{metric}:\t\t\t{val}")
    print('\n')

In [33]:
import json

def save_json(name: str, data: typ.Dict):
    with open(f'{name}.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def load_json(name: str):
    with open(name, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

### Результаты  baseline

In [ ]:
LOAD_RESULTS = True


baseline_model_results: typ.Dict[str, typ.Dict] = dict() 
if not LOAD_RESULTS:
    for model_id in WHISPER_MODELS:
        model, processor = load_model(model_id)
        res = evaluate(model, processor, dataset=mini_dataset)

        baseline_model_results[model_id] = res
        print_model_result(model_id, res)

    save_json("baseline_model_results", baseline_model_results)
    clear_output(wait=True)
    
baseline_model_results = load_json("baseline_model_results.json")
baseline_models_df = pd.DataFrame(baseline_model_results)
baseline_models_df

,openai/whisper-tiny,openai/whisper-base,openai/whisper-small,openai/whisper-medium,openai/whisper-large-v3
rtf,0.071634,0.120351,0.232859,0.621033,1.313134
wer,0.613821,0.520325,0.373984,0.373984,0.292683
cer,0.297281,0.284920,0.231768,0.260816,0.196539
total_duration,249.197937,249.197937,249.197937,249.197937,249.197937
total_time,17.851038,29.991146,58.028057,154.760191,327.230241
num_samples,50.000000,50.000000,50.000000,50.000000,50.000000


### structural pruning

будем удалять нейроны в линейных слоях и свёртках, профилировщик показал, что они занимают больше всего времени

In [26]:
import torch_pruning as tp

def structural_iterative_pruning(
        model,
        processor,
        importance,
        ignored_layers,
        sample,
        pruning_ratio: float=0.5,
        steps: int = 1,
        round_to: int= 8,
        global_pruning=True,
        device = "cpu",
        print_results: bool = True
) -> typ.Dict[int, typ.Dict]:
    model.eval()
    inputs, _, _ = process_sample(sample, processor, device, model.dtype)

    batch_size = inputs["input_features"].shape[0]
    inputs["decoder_input_ids"] = torch.full(
        (batch_size, 1),
        model.config.decoder_start_token_id,
        dtype=torch.long,
        device=device
    )

    pruner = tp.pruner.MetaPruner(
        model,
        example_inputs=inputs,
        importance=importance,
        forward_fn=lambda m, x: m(**x),
        output_transform=lambda out: out.logits,
        ignored_layers=ignored_layers,
        pruning_ratio=pruning_ratio,
        iterative_steps=steps,
        round_to=round_to,
        global_pruning=global_pruning
    )

    results: typ.Dict[int, typ.Dict] = dict()
    with torch.no_grad():
        for i in range(4):
            pruner.step()
            pr_res = evaluate(model, processor, mini_dataset, device=device)
            results[i + 1] = pr_res

            if print_results:
                print_model_result(f"iteration_{i}", pr_res)

    return results

### Unstructured magnitude pruning

In [28]:
def magnitude_pruning(model: torch.nn.Module, amount: float = 0.2):
    def _model_pruning(model):
        for module in model.modules():
            if isinstance(module, (torch.nn.Linear, torch.nn.Conv1d)) and prune.is_pruned(module):
                prune.remove(module, 'weight')
                
    for module in model.modules():
        if isinstance(module, (torch.nn.Linear, torch.nn.Conv1d)):
            prune.l1_unstructured(module, name="weight", amount=amount)
            
    _model_pruning(model)
    return model


### Результаты после unstructured pruning

In [ ]:
LOAD_RESULTS = True

unstr_pruning_results = dict()
if not LOAD_RESULTS:
    for model_id in WHISPER_MODELS:
        for amount in (0.1, 0.2, 0.25, 0.3):
            model, processor = load_model(model_id)
            model = magnitude_pruning(model, amount)

            pr_res = evaluate(model, processor, mini_dataset)
            unstr_pruning_results[f'{model_id}[unstr_mg_pr_{amount}]'] = pr_res
            print_model_result(f'{model_id}[unstr_mg_pr_{amount}]', pr_res)
    save_json("unstr_mg_pr_results", unstr_pruning_results)
    
clear_output(wait=True)

unstr_pruning_results = load_json("unstr_mg_pr_results.json")
unstr_pruning_df = pd.DataFrame(unstr_pruning_results)
unstr_pruning_df


,openai/whisper-tiny[unstr_mg_pr_0.1],openai/whisper-tiny[unstr_mg_pr_0.2],openai/whisper-tiny[unstr_mg_pr_0.3],openai/whisper-base[unstr_mg_pr_0.1],openai/whisper-base[unstr_mg_pr_0.2],openai/whisper-base[unstr_mg_pr_0.3],openai/whisper-small[unstr_mg_pr_0.1],openai/whisper-tiny[unstr_mg_pr_0.25],openai/whisper-base[unstr_mg_pr_0.25],openai/whisper-small[unstr_mg_pr_0.2],openai/whisper-small[unstr_mg_pr_0.25],openai/whisper-small[unstr_mg_pr_0.3],openai/whisper-medium[unstr_mg_pr_0.1],openai/whisper-medium[unstr_mg_pr_0.2],openai/whisper-medium[unstr_mg_pr_0.25],openai/whisper-medium[unstr_mg_pr_0.3],openai/whisper-large-v3[unstr_mg_pr_0.1],openai/whisper-large-v3[unstr_mg_pr_0.2],openai/whisper-large-v3[unstr_mg_pr_0.25],openai/whisper-large-v3[unstr_mg_pr_0.3]
rtf,0.067493,0.068577,0.225660,0.121723,0.116623,0.146901,0.229630,0.138237,0.106617,0.226232,0.228875,0.220751,0.624964,0.638332,0.604593,0.601354,1.080871,1.034719,1.023045,1.047473
wer,0.593496,0.630081,2.589431,0.544715,0.565041,0.674797,0.382114,1.300813,0.577236,0.373984,0.373984,0.382114,0.378049,0.369919,0.361789,0.349593,0.288618,0.288618,0.296748,0.280488
cer,0.300371,0.295426,1.854759,0.297281,0.294808,0.608158,0.241038,0.855995,0.278739,0.225587,0.239184,0.239802,0.257726,0.244747,0.228059,0.232386,0.192213,0.186650,0.200865,0.195303
total_duration,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937,249.197937
total_time,16.819158,17.089338,56.234107,30.333196,29.062175,36.607369,57.223253,34.448330,26.568746,56.376545,57.035299,55.010744,155.739819,159.070916,150.663302,149.856239,269.350713,257.849765,254.940686,261.028222
num_samples,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
